# 05 · Inferência batch

*Pré-calcular o top-10 de cada usuário — como o mercado faz*

> **FIAP Pós-Tech · Fase 5 (MLOps)** — Recomendação de notícias com MLOps no Databricks.
> Matérias **reais** (sitemap do O Globo) · usuários e cliques **sintéticos**.

## Batch ou tempo real?

| | **Batch** (nosso) | **Tempo real** (Model Serving) |
|---|---|---|
| Como | Job calcula o top-10 de todos e grava numa tabela | API calcula a cada requisição |
| Latência no app | só uma leitura (ms) | inferência a cada clique |
| Frescor | atualiza a cada execução (diária/horária) | instantâneo |
| Custo/complexidade | baixo | endpoint sempre de pé, autoscaling |
| Quando usar | home personalizada, e-mail, push | "porque você acabou de ler X" |

Muitos portais combinam os dois: **candidatos em batch + reordenação em tempo real**.

In [ ]:
# Célula 1 — dependências (sempre a primeira: o restartPython apaga as variáveis)
%pip install -q -r ../requirements-databricks.txt
dbutils.library.restartPython()

In [ ]:
# Célula 2 — src/ no sys.path + widgets (= parâmetros do Job) -> settings
import sys
from pathlib import Path


def _find_src() -> Path:
    candidates = [Path.cwd().parent / "src"]
    try:  # fallback: caminho do notebook no workspace
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        candidates.append(Path("/Workspace" + ctx.notebookPath().get()).parent.parent / "src")
    except Exception:  # noqa: BLE001
        pass
    for c in candidates:
        if (c / "news_recsys").is_dir():
            return c.resolve()
    raise FileNotFoundError(f"src/ não encontrado em {candidates}")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
ROOT = SRC.parent

from news_recsys.config import Settings

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "news_recsys_dev")
dbutils.widgets.text("n_users", "300")
dbutils.widgets.text("seed", "42")

settings = Settings(_env_file=None).with_overrides(
    catalog=dbutils.widgets.get("catalog"),
    schema_name=dbutils.widgets.get("schema"),
    n_users=dbutils.widgets.get("n_users"),
    seed=dbutils.widgets.get("seed"),
)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {settings.catalog}.{settings.schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {settings.catalog}.{settings.schema_name}.{settings.volume}")
print(f"schema ..... {settings.catalog}.{settings.schema_name}")
print(f"modelo ..... {settings.model_fqn}")
print(f"usuários ... {settings.n_users} (seed={settings.seed})")

In [ ]:
# MLflow: experimento por usuário + registro de modelos no Unity Catalog
import mlflow
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
ME = w.current_user.me().user_name
EXPERIMENT = f"/Users/{ME}/news_recsys_{settings.schema_name}"
mlflow.set_experiment(EXPERIMENT)
mlflow.set_registry_uri("databricks-uc")
print("experimento:", EXPERIMENT)

In [ ]:
# Dados de entrada: catálogo (bronze) + usuários e cliques (silver)
import pandas as pd

articles = spark.table(settings.table("bronze_articles")).toPandas()
window = spark.table(settings.table("silver_catalog_window")).toPandas()["article_id"]
articles = articles[articles["article_id"].isin(window)].reset_index(drop=True)
users = spark.table(settings.table("silver_users")).toPandas()
clicks = spark.table(settings.table("silver_interactions")).toPandas()
clicks["event_ts"] = pd.to_datetime(clicks["event_ts"], utc=True)
print(f"{len(articles)} matérias · {len(users)} usuários · {len(clicks)} cliques")

### Execução: carregando o `@champion` e recomendando para todos

In [ ]:
from datetime import UTC, datetime

from mlflow import MlflowClient

champion = MlflowClient(registry_uri="databricks-uc").get_model_version_by_alias(settings.model_fqn, "champion")
model = mlflow.pyfunc.load_model(f"models:/{settings.model_fqn}@champion")
print(f"usando {settings.model_fqn} versão {champion.version} (@champion)")

recs = model.predict(pd.DataFrame({"user_id": users["user_id"]}), params={"k": settings.k})
meta = articles[["article_id", "title", "url", "section", "image_url", "published_at"]]
gold = recs.merge(meta, on="article_id", how="left").assign(
    model_version=str(champion.version), generated_at=datetime.now(UTC).isoformat(timespec="seconds")
)
print(f"{len(gold)} recomendações para {gold['user_id'].nunique()} usuários")
gold.head(10)[["user_id", "rank", "section", "title", "score"]]

### Execução: gravando `gold_user_recommendations`

In [ ]:
GOLD = settings.table("gold_user_recommendations")
spark.createDataFrame(gold).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD)
print(GOLD, "->", spark.table(GOLD).count(), "linhas")
spark.sql(f"SELECT model_version, generated_at, count(*) AS recs, count(DISTINCT user_id) AS usuarios FROM {GOLD} GROUP BY ALL").toPandas()

### Faz sentido? Um usuário de cada persona

O que o usuário leu (editorias) × o que recomendamos. É a "checagem de bom senso" que
nenhuma métrica substitui.

In [ ]:
section = articles.set_index("article_id")["section"]
sample = users.groupby("persona").head(1)
rows = []
for u in sample.itertuples():
    read = clicks.loc[clicks["user_id"] == u.user_id, "article_id"].map(section).value_counts().head(2)
    rec = gold.loc[gold["user_id"] == u.user_id, "section"].value_counts().head(2)
    rows.append({"usuário": u.user_id, "persona": u.persona, "leu (top editorias)": dict(read), "recomendado": dict(rec)})
pd.DataFrame(rows)

### O que percebemos?

- A inferência usa **`@champion`**, não "a última versão" → o gate do notebook 04 protege o app.
- A tabela gold carrega `model_version`: dá para saber **qual modelo** gerou cada recomendação.
- O app Streamlit só faz `SELECT ... WHERE user_id = ?` nesta tabela (via SQL warehouse).

**Próximo:** `06_monitoramento`.